In [1]:
from google.colab import userdata
##userdata.get('HF_token')

ModuleNotFoundError: No module named 'google.colab'

In [2]:
%%capture

# Core training libraries
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    tokenizers==0.19.1 \
    accelerate==0.34.2 \
    peft==0.12.0 \
    trl==0.9.6 \
    bitsandbytes==0.43.1 \
    evaluate==0.4.2

# Utilities
!pip install -q \
    pandas \
    scikit-learn \
    rich \
    pyyaml \
    python-dotenv \
    tqdm



# Evaluation (requires pydantic v2)
!pip install -q --upgrade pydantic
!pip install -q google-genai rouge-score

print("✅ Installation complete!")
print("✅ All dependencies compatible (pydantic v2 + google-genai)")

In [4]:
import sys
import torch

print("="*60)
print("ENVIRONMENT CHECK")
print("="*60)
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ WARNING: CUDA not available. Training will be VERY slow on CPU.")

print("="*60)

ENVIRONMENT CHECK
Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA version: 12.6
Device name: Tesla T4
Device capability: (7, 5)
Total VRAM: 14.74 GB


In [5]:
# Reinstall numpy to fix potential compatibility issues
!pip install --upgrade --force-reinstall numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.90 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.90 requires numpy>=2; python_versi

In [6]:
import os
import random
import numpy as np
import torch

SEED = 42

# Set environment variable for Python hash seed
os.environ['PYTHONHASHSEED'] = str(SEED)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Note: These settings may impact performance
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"✅ Seeds set to {SEED} for reproducibility")
print("⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations")

✅ Seeds set to 42 for reproducibility
⚠️ Note: Full determinism on GPU is not guaranteed due to non-deterministic operations


In [7]:
os.environ["HF_TOKEN"] = userdata.get('HF_token')
!hf auth login --token $HF_TOKEN

print("ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.")

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Bootcamp_FineTuning` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
ℹ️ Hugging Face login skipped. Uncomment login() to push models to Hub.


In [8]:
import torch
from pprint import pprint

# Auto-detect compute dtype (BF16 requires compute capability >= 8.0)
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

CONFIG = {
    # Model
    "base_model": "Qwen/Qwen2.5-1.5B-Instruct",
    # Alternative for tighter VRAM: "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    # For GGUF export, prefer: "meta-llama/Llama-3.2-3B-Instruct" or Mistral models

    # Dataset
    "dataset_name": "lavita/AlpaCare-MedInstruct-52k",
    "dataset_split": "train",
    "dataset_subsample": 500,  # Colab-safe: 500 | Local: 1500
    "train_val_split": 0.9,  # 90% train, 10% validation

    # Tokenization
    "max_length": 512,  # Colab: 512 | Local: 1024

    # Training
    "num_train_epochs": 1,
    "max_steps": 250,  # Colab: 250 | Local: 600
    "per_device_train_batch_size": 1,  # Colab: 1 | Local: 2
    "gradient_accumulation_steps": 64,  # Colab: 64 | Local: 32
    "learning_rate": 2e-5,
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 200,
    "eval_steps": 100,
    "save_total_limit": 2,

    # LoRA
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    # Quantization
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": compute_dtype,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # Output
    "output_dir": "outputs/adapter",
    "push_to_hub": False,

    # Generation
    "max_new_tokens": 128,
    "temperature": 0.0,  # Deterministic
    "do_sample": True,

    # HF credentials
    'hf_username': '<your-username>',
    'hub_model_name': 'zuucrew-medical-assistant',
}

# Effective batch size
effective_batch_size = CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]

print("="*60)
print("CONFIGURATION (COLAB FREE TIER)")
print("="*60)
pprint(CONFIG)
print("="*60)
print(f"Compute dtype: {compute_dtype}")
print(f"Using BF16: {use_bf16}")
print(f"Effective batch size: {effective_batch_size}")
print("="*60)

CONFIGURATION (COLAB FREE TIER)
{'base_model': 'Qwen/Qwen2.5-1.5B-Instruct',
 'bnb_4bit_compute_dtype': torch.float16,
 'bnb_4bit_quant_type': 'nf4',
 'bnb_4bit_use_double_quant': True,
 'dataset_name': 'lavita/AlpaCare-MedInstruct-52k',
 'dataset_split': 'train',
 'dataset_subsample': 500,
 'do_sample': True,
 'eval_steps': 100,
 'gradient_accumulation_steps': 64,
 'hf_username': '<your-username>',
 'hub_model_name': 'zuucrew-medical-assistant',
 'learning_rate': 2e-05,
 'load_in_4bit': True,
 'logging_steps': 10,
 'lora_alpha': 32,
 'lora_dropout': 0.05,
 'lora_r': 16,
 'lora_target_modules': ['q_proj',
                         'k_proj',
                         'v_proj',
                         'o_proj',
                         'gate_proj',
                         'up_proj',
                         'down_proj'],
 'max_length': 512,
 'max_new_tokens': 128,
 'max_steps': 250,
 'num_train_epochs': 1,
 'output_dir': 'outputs/adapter',
 'per_device_train_batch_size': 1,
 'push_to_hub

In [9]:
from datasets import load_dataset, Dataset
import json

def load_medical_dataset(dataset_name, split, subsample, seed=42):
    """Load dataset with robust field mapping and fallback."""

    try:
        # Try loading from Hugging Face
        print(f"📥 Loading dataset: {dataset_name}...")
        dataset = load_dataset(dataset_name, split=split)
        dataset = dataset.shuffle(seed=seed).select(range(min(subsample, len(dataset))))
        print(f"✅ Loaded {len(dataset)} examples from Hugging Face")

    except Exception as e:
        print(f"⚠️ Failed to load from Hugging Face: {e}")
        print("🔄 Creating synthetic fallback dataset...")

        # Create synthetic medical instruction data
        synthetic_data = []
        templates = [
            {
                "instruction": "Explain the following medical term in simple language.",
                "input": "Hypertension",
                "output": "Hypertension, commonly known as high blood pressure, is a condition where the force of blood against artery walls is consistently too high. This can lead to serious health complications if left untreated."
            },
            {
                "instruction": "What are the common symptoms of the following condition?",
                "input": "Type 2 Diabetes",
                "output": "Common symptoms of Type 2 Diabetes include increased thirst, frequent urination, increased hunger, fatigue, blurred vision, slow-healing sores, and frequent infections."
            },
            {
                "instruction": "Provide general advice for managing the following health issue.",
                "input": "Chronic back pain",
                "output": "Managing chronic back pain typically involves: maintaining good posture, regular low-impact exercise like swimming or walking, maintaining a healthy weight, using proper lifting techniques, and consulting with healthcare providers for appropriate treatment options."
            },
        ]

        # Duplicate to reach ~120 examples
        for i in range(40):
            for template in templates:
                synthetic_data.append(template)

        # Save to temporary JSONL
        with open("/tmp/synthetic_medical.jsonl", "w") as f:
            for item in synthetic_data[:subsample]:
                f.write(json.dumps(item) + "\n")

        dataset = load_dataset("json", data_files="/tmp/synthetic_medical.jsonl", split="train")
        print(f"✅ Created synthetic dataset with {len(dataset)} examples")

    return dataset


def map_dataset_fields(example):
    """Robustly map dataset fields to instruction/input/output schema."""

    # Try to find instruction
    instruction = None
    for key in ["instruction", "question", "prompt", "task"]:
        if key in example and example[key]:
            instruction = str(example[key]).strip()
            break

    # Try to find input (optional)
    input_text = ""
    for key in ["input", "context", "passage", "history"]:
        if key in example and example[key]:
            input_text = str(example[key]).strip()
            break

    # Try to find output/target
    output = None
    for key in ["output", "response", "answer", "target", "completion"]:
        if key in example and example[key]:
            output = str(example[key]).strip()
            break

    return {
        "instruction": instruction,
        "input": input_text,
        "output": output
    }


# Load dataset
dataset = load_medical_dataset(
    CONFIG["dataset_name"],
    CONFIG["dataset_split"],
    CONFIG["dataset_subsample"],
    seed=SEED
)

print(f"\n📊 Dataset before cleaning: {len(dataset)} examples")

# Map fields
dataset = dataset.map(map_dataset_fields)

# Drop rows with missing instruction or output
dataset = dataset.filter(lambda x: x["instruction"] is not None and x["output"] is not None)

print(f"📊 Dataset after cleaning: {len(dataset)} examples")
print(f"✅ Dropped {CONFIG['dataset_subsample'] - len(dataset)} examples with missing data\n")

# Split into train/validation
split_dataset = dataset.train_test_split(
    train_size=CONFIG["train_val_split"],
    seed=SEED
)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print(f"📊 Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("\n📝 Sample example:")
print(train_dataset[0])

📥 Loading dataset: lavita/AlpaCare-MedInstruct-52k...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

✅ Loaded 500 examples from Hugging Face

📊 Dataset before cleaning: 500 examples


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

📊 Dataset after cleaning: 500 examples
✅ Dropped 0 examples with missing data

📊 Train: 450 | Validation: 50

📝 Sample example:
{'output': "As a 40-year-old pregnant woman, your age does increase the risk of having a baby with Down syndrome. However, it's important to note that the majority of babies born to women in their 40s are still healthy and do not have Down syndrome. \n\nThe risk of having a baby with Down syndrome at the age of 40 is approximately 1 in 100. This means that out of 100 pregnancies at this age, around 1 will be affected by Down syndrome. \n\nTo get more accurate information about your individual risk, you may consider undergoing prenatal screening or diagnostic tests. These tests can provide more specific information regarding the chance of your baby having Down syndrome. It's advisable to consult with your healthcare provider who can guide you through the appropriate testing options based on your personal medical history and preferences.", 'input': '<noinput>', 

In [10]:
import pandas as pd

# Convert first 50 samples to dataframe
df_preview = pd.DataFrame(train_dataset[:50])

# Display with formatting
pd.set_option('display.max_colwidth', 100)  # Limit column width for readability
print(f"📊 Displaying first 50 samples out of {len(dataset)} total examples\n")
df_preview

📊 Displaying first 50 samples out of 500 total examples



,output,input,instruction
0,"As a 40-year-old pregnant woman, your age does increase the risk of having a baby with Down synd...",<noinput>,"Ask about the possible genetic risks your child might face related to Down Syndrome, given that ..."
1,"As a medical expert, I cannot provide a specific treatment recommendation without evaluating the...","Patient information: 55 years old female, with a known family history of essential hypertension ...","Based on the given medical history, which treatment option for essential hypertension would be b..."
2,The heart's electrical system plays a crucial role in making the heart beat and ensuring the con...,"The heart's electrical wiring keeps it beating, which controls the continuous exchange of oxygen...",Simplify the explanation about heart's electrical system and its role in making the heart beat.
3,Chemotherapy is a common treatment for breast cancer and can be effective in destroying cancer c...,I got diagnosed with breast cancer and my doctor said I need chemotherapy. I'm worried about the...,Discuss your concerns about chemotherapy's side effects with an oncologist.
4,Pneumonia is an infection that causes inflammation in the small air sacs called alveoli in one o...,"""Pneumonia is an infection that inflames the alveoli in one or both lungs.",Simplify the following complex medical term into simpler terminologies.
5,"Based on the symptoms and history provided, there are several possible diagnoses to consider. Th...","Patient is 45 female, shortness of breath especially on laying down, fatigue, lower ankle swelli...","Based on the symptoms and history provided, make a probable diagnosis considering multiple disea..."
6,The major type of muscle present at the back region is C) Skeletal muscles.,A) Smooth muscles B) Cardiac muscles C) Skeletal muscles D) Pharyngeal muscle,Choose the major type of muscle present at the back region.
7,Heart failure develops over time as a result of various underlying conditions and factors. Initi...,<noinput>,Write a comprehensive paragraph explaining how heart failure develops over time.
8,How does your muscular system work when you lift a heavy object?,<noinput>,Ask a question related to how your muscular system works when you lift a heavy object.
9,To manage high blood glucose levels in a patient with type-2 diabetes mellitus who is already on...,A 60-year-old man with a history of type-2 diabetes mellitus is using Metformin. Upon routine ch...,"Solve the following USMLE-style question, providing the correct answer supported by reasoning."


In [11]:
from transformers import AutoTokenizer
import numpy as np

# Load tokenizer for diagnostics
print(f"Loading tokenizer: {CONFIG['base_model']}")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)

# Ensure pad token is set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Sample up to 500 examples for diagnostics
sample_size = min(500, len(train_dataset))
sample_dataset = train_dataset.select(range(sample_size))

# Compute token lengths
token_lengths = []
for example in sample_dataset:
    # Concatenate instruction + input + output
    text = f"{example['instruction']} {example['input']} {example['output']}"
    tokens = tokenizer(text, add_special_tokens=True)
    token_lengths.append(len(tokens["input_ids"]))

token_lengths = np.array(token_lengths)

print("="*60)
print("TOKEN LENGTH DIAGNOSTICS")
print("="*60)
print(f"Sample size: {sample_size}")
print(f"Average token length: {token_lengths.mean():.1f}")
print(f"Median token length: {np.median(token_lengths):.1f}")
print(f"Min token length: {token_lengths.min()}")
print(f"Max token length: {token_lengths.max()}")
print(f"95th percentile: {np.percentile(token_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(token_lengths, 99):.1f}")
print()
truncated = (token_lengths > CONFIG["max_length"]).sum()
truncation_rate = truncated / len(token_lengths) * 100
print(f"Truncation at max_length={CONFIG['max_length']}: {truncated}/{len(token_lengths)} ({truncation_rate:.1f}%)")
print("="*60)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Loading tokenizer: Qwen/Qwen2.5-1.5B-Instruct


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

TOKEN LENGTH DIAGNOSTICS
Sample size: 450
Average token length: 233.0
Median token length: 265.0
Min token length: 28
Max token length: 496
95th percentile: 363.5
99th percentile: 423.4

Truncation at max_length=512: 0/450 (0.0%)
